# 018 Testing Whether Particular AA Model Orders Concentrate PCA-Equivalent Information

## Hypothesis

> Certain parent AA model orders concentrate PCA-equivalent information into unusually small sets of informative archetypes.

The previous PCA benchmark established a conventional PCA information-retention curve and, for each sparse MS-AA top-\(m\) reconstruction, estimated the number of PCA dimensions required to reach comparable cross-subject timepoint-identification accuracy.

This notebook formalizes that idea.

For each MS-AA parent model order \(K\) and retained subset size \(m\), we estimate a continuous PCA-equivalent dimensionality,

\[
q_{\mathrm{eq}}(K,m),
\]

and define the **information-concentration ratio**

\[
C(K,m)=\frac{q_{\mathrm{eq}}(K,m)}{m}.
\]

Interpretation:

- \(C>1\): \(m\) selected archetypes preserve as much timepoint information as **more than \(m\)** conventional PCA dimensions.
- \(C\approx1\): similar information density.
- \(C<1\): PCA is more compact in the conventional information-preservation sense.

The central question is **not** simply whether AA beats PCA. Instead:

> Does the parent archetypal geometry at particular \(K\) values cause information to become unusually concentrated in a small number of archetypes?

## Main analyses

1. Compute continuous PCA-equivalent dimensionality by interpolation on the monotonic PCA performance envelope.
2. Compute \(C=q_{\mathrm{eq}}/m\) and \(\log_2 C\).
3. Plot concentration across parent AA \(K\).
4. Aggregate concentration across top-\(m\) values.
5. Test whether particular \(K\) values show stronger concentration than expected under permutation of parent-\(K\) labels.
6. Test pre-specified candidate regimes such as spatial \(K=50\), without selecting them from the same concentration statistic.
7. Quantify local enhancement relative to neighboring \(K\) values.
8. Compare spatial and temporal AA using normalized parent-component ratio.
9. Produce manuscript-ready summary tables.

## Important limitation

The current top-\(m\) AA subsets were selected using decoding performance on the same data used for evaluation. Therefore these analyses are currently **descriptive/exploratory evidence for concentration structure**. The strongest confirmatory version should be rerun after top-\(m\) archetype ranking is made held-out.


In [ ]:
# ============================================================
# SETTINGS
# ============================================================

from pathlib import Path

FIT_SCOPE = "across"

# ------------------------------------------------------------
# Existing outputs
# ------------------------------------------------------------

PCA_OUTPUT_DIR = Path("pca_aa_equivalence_outputs")
PCA_SUMMARY_CSV = PCA_OUTPUT_DIR / "pca_summary.csv"

AA_OUTPUT_TEMPLATE = "msaa_condrank_decoding_outputs_{analysis_type}_across"
AA_ANALYSIS_TYPES = ["spatial", "temporal"]

# ------------------------------------------------------------
# Dimensions
# ------------------------------------------------------------

SPATIAL_DENOMINATOR = 700
TEMPORAL_DENOMINATOR = 300

# ------------------------------------------------------------
# Conditions / subset sizes
# ------------------------------------------------------------

CONDITIONS = ["intact", "word", "rest"]

# Main sparse range. These should already exist in notebook 002.
TOP_M_VALUES = [3, 5, 7, 10, 15]

# For the primary paper figure, probably focus on intact and word.
PRIMARY_CONDITIONS = ["intact", "word"]

# ------------------------------------------------------------
# Pre-specified candidate K values
# ------------------------------------------------------------
# These are based on the PREVIOUS AA analyses, not selected from
# the concentration statistic in this notebook.

CANDIDATE_K_BY_ANALYSIS = {
    "spatial": [50, 70, 88],
    "temporal": [14, 15, 23],
}

# Main pre-specified spatial candidate motivated by earlier top-m bump
PRIMARY_SPATIAL_K = 50

# ------------------------------------------------------------
# Permutation testing
# ------------------------------------------------------------

N_PERM = 10000
RNG_SEED = 42

# We test:
# - maximum K-level concentration (global "is there a special K?" test)
# - specific pre-specified K values
#
# K labels are permuted WITHIN condition x top_m x analysis_type,
# preserving the marginal distribution of concentration values.

# ------------------------------------------------------------
# Aggregation
# ------------------------------------------------------------

# log2 concentration is the preferred scale:
# 0 = PCA-equivalent information density
# >0 = AA subset corresponds to more PCA dimensions than archetypes
# <0 = AA subset corresponds to fewer PCA dimensions than archetypes

AGGREGATE_SCORE = "log2_concentration"

# Minimum number of m-values needed to retain a K in aggregate plots
MIN_M_PER_K = 2

# ------------------------------------------------------------
# Outputs
# ------------------------------------------------------------

OUTPUT_DIR = Path("aa_information_concentration_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIG_ROOT = Path("outputs/notebook_figures")
FIG_DIR = FIG_ROOT / "018_AA_information_concentration"
FIG_DIR.mkdir(parents=True, exist_ok=True)

SAVE_FIGS = True
FIG_FORMAT = "pdf"
DPI = 300

print("Output directory:", OUTPUT_DIR.resolve())
print("Figure directory:", FIG_DIR)


In [ ]:
# ============================================================
# IMPORTS / HELPERS
# ============================================================

%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import spearmanr

plt.rcParams.update({
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

_fig_counter = 0

def _safe_name(name):
    return "".join(
        c if c.isalnum() or c in "_-." else "_"
        for c in str(name)
    )[:180]

def save_current_fig(name):
    global _fig_counter

    if not SAVE_FIGS:
        return None

    _fig_counter += 1

    out = FIG_DIR / f"{_fig_counter:03d}_{_safe_name(name)}.{FIG_FORMAT}"

    plt.savefig(
        out,
        dpi=DPI,
        bbox_inches="tight",
    )

    print("Saved:", out)

    return out

def analysis_denominator(analysis_type):
    return (
        SPATIAL_DENOMINATOR
        if analysis_type == "spatial"
        else TEMPORAL_DENOMINATOR
    )


## 1. Load conventional PCA and sparse MS-AA results

In [ ]:
# ============================================================
# LOAD PCA
# ============================================================

if "pca_summary_df" in globals():
    pca_df = pca_summary_df.copy()
    print("Using pca_summary_df already in memory.")
else:
    if not PCA_SUMMARY_CSV.exists():
        raise FileNotFoundError(
            f"Could not find {PCA_SUMMARY_CSV}. "
            "Run notebook 017 first, or update PCA_SUMMARY_CSV."
        )

    pca_df = pd.read_csv(
        PCA_SUMMARY_CSV
    )

    print("Loaded:", PCA_SUMMARY_CSV)

# Direct AA comparison uses PCA reconstruction-space accuracy.
pca_df = pca_df[
    pca_df["decode_space"].astype(str)
    == "reconstruction"
].copy()

pca_df["K"] = pca_df["K"].astype(int)
pca_df["condition"] = pca_df["condition"].astype(str)

display(pca_df.head())


In [ ]:
# ============================================================
# LOAD AA TOP-M
# ============================================================

aa_parts = []

for analysis_type in AA_ANALYSIS_TYPES:

    path = (
        Path(
            AA_OUTPUT_TEMPLATE.format(
                analysis_type=analysis_type
            )
        )
        / "topm_summary.csv"
    )

    if not path.exists():
        print("Missing:", path)
        continue

    x = pd.read_csv(path)

    if "analysis_type" not in x.columns:
        x["analysis_type"] = analysis_type

    x["analysis_type"] = x["analysis_type"].astype(str)
    x["condition"] = x["condition"].astype(str)
    x["K"] = x["K"].astype(int)
    x["top_m"] = x["top_m"].astype(int)

    x["parent_ratio"] = (
        x["K"]
        / analysis_denominator(analysis_type)
    )

    aa_parts.append(x)

    print("Loaded:", path)

if not aa_parts:
    raise FileNotFoundError(
        "No AA topm_summary.csv files were found."
    )

aa_topm = pd.concat(
    aa_parts,
    ignore_index=True,
)

aa_topm = aa_topm[
    aa_topm["top_m"].isin(TOP_M_VALUES)
].copy()

display(aa_topm.head())
print("AA rows:", len(aa_topm))


## 2. Estimate continuous PCA-equivalent dimensionality

The earlier notebook used the first tested PCA \(K\) whose monotonic-envelope performance reached the AA target. That is intuitive, but discretized by the PCA grid.

Here we improve this by linearly interpolating the **monotonic PCA accuracy envelope**.

For each condition:

1. sort PCA results by \(K\);
2. replace empirical accuracy by its cumulative maximum;
3. collapse flat duplicate accuracy values;
4. interpolate the PCA dimension corresponding to each AA accuracy.

This gives a continuous \(q_{\mathrm{eq}}\).


In [ ]:
# ============================================================
# CONTINUOUS PCA-EQUIVALENT DIMENSION
# ============================================================

def monotonic_pca_envelope(condition):

    p = pca_df[
        pca_df["condition"]
        == condition
    ].copy()

    p = p.sort_values(
        "K"
    ).reset_index(drop=True)

    p["accuracy_envelope"] = (
        p["mean"].cummax()
    )

    return p

def interpolate_pca_equivalent_dimension(
    target_accuracy,
    condition,
):
    """
    Continuous PCA-equivalent dimensionality from the monotonic
    reconstruction-space PCA accuracy curve.
    """

    p = monotonic_pca_envelope(
        condition
    )

    ks = p["K"].to_numpy(
        dtype=float
    )

    acc = p[
        "accuracy_envelope"
    ].to_numpy(dtype=float)

    # Below first tested PCA accuracy:
    # interpolate from chance at K=0 to first PCA point.
    chance = (
        float(p["chance"].iloc[0])
        if "chance" in p.columns
        else 1.0 / 300.0
    )

    if target_accuracy <= acc[0]:

        if acc[0] <= chance:
            return 0.0

        frac = (
            (target_accuracy - chance)
            / (acc[0] - chance)
        )

        return float(
            np.clip(
                frac,
                0,
                1,
            )
            * ks[0]
        )

    # Above best PCA performance
    if target_accuracy > acc[-1]:
        return np.nan

    # Find first envelope point reaching target
    hi = np.searchsorted(
        acc,
        target_accuracy,
        side="left",
    )

    lo = hi - 1

    a0, a1 = acc[lo], acc[hi]
    k0, k1 = ks[lo], ks[hi]

    # Flat envelope segment
    if np.isclose(a1, a0):
        return float(k1)

    frac = (
        target_accuracy - a0
    ) / (
        a1 - a0
    )

    return float(
        k0
        + frac * (k1 - k0)
    )

# Sanity check
for condition in CONDITIONS:
    p = monotonic_pca_envelope(condition)

    print(
        condition,
        "PCA envelope max =",
        p["accuracy_envelope"].max(),
        "at/after K =",
        p.loc[
            p["accuracy_envelope"].idxmax(),
            "K",
        ],
    )


In [ ]:
# ============================================================
# BUILD CONCENTRATION TABLE
# ============================================================

rows = []

for _, r in aa_topm.iterrows():

    q_eq = interpolate_pca_equivalent_dimension(
        target_accuracy=float(r["mean"]),
        condition=str(r["condition"]),
    )

    m = int(r["top_m"])

    if np.isfinite(q_eq) and m > 0:

        concentration = (
            q_eq / m
        )

        log2_concentration = (
            np.log2(concentration)
            if concentration > 0
            else np.nan
        )

    else:

        concentration = np.nan
        log2_concentration = np.nan

    rows.append({
        "analysis_type": str(
            r["analysis_type"]
        ),
        "condition": str(
            r["condition"]
        ),
        "K": int(
            r["K"]
        ),
        "parent_ratio": float(
            r["parent_ratio"]
        ),
        "top_m": m,
        "AA_accuracy": float(
            r["mean"]
        ),
        "AA_err": (
            float(r["err"])
            if "err" in r.index
            else np.nan
        ),
        "pca_equivalent_dimension": q_eq,
        "concentration_ratio": concentration,
        "log2_concentration": log2_concentration,
    })

concentration_df = pd.DataFrame(
    rows
)

concentration_df.to_csv(
    OUTPUT_DIR / "AA_PCA_information_concentration_long.csv",
    index=False,
)

display(
    concentration_df
    .sort_values(
        "log2_concentration",
        ascending=False,
    )
    .head(30)
)


## 3. First descriptive test: does concentration exceed 1?

This is not yet a test of a special parent \(K\). It asks the simpler question:

> Are there AA configurations for which \(m\) archetypes carry as much information as more than \(m\) PCA dimensions?

The horizontal reference line \(C=1\) is therefore important.


In [ ]:
# ============================================================
# CONCENTRATION BY PARENT K AND TOP-M
# ============================================================

def plot_concentration_by_K(
    analysis_type="spatial",
    condition="intact",
):
    sub = concentration_df[
        (concentration_df["analysis_type"] == analysis_type) &
        (concentration_df["condition"] == condition)
    ].copy()

    if len(sub) == 0:
        return

    fig, ax = plt.subplots(
        figsize=(8.5, 5.2)
    )

    ax.axhline(
        1.0,
        color="gray",
        linestyle="--",
        linewidth=1.2,
        label="PCA-equivalent density (C=1)",
    )

    for m in sorted(
        sub["top_m"].unique()
    ):

        s = sub[
            sub["top_m"] == m
        ].sort_values("K")

        ax.plot(
            s["K"],
            s["concentration_ratio"],
            marker="o",
            markersize=3,
            linewidth=1.5,
            label=f"top-{m}",
        )

    ax.set_xlabel(
        "Parent MS-AA model order K"
    )

    ax.set_ylabel(
        "PCA-equivalent information concentration\n"
        r"$C=q_{\mathrm{eq}}/m$"
    )

    ax.set_title(
        f"{analysis_type} MS-AA | {condition}"
    )

    ax.legend(
        frameon=False,
        ncol=2,
    )

    plt.tight_layout()

    save_current_fig(
        f"concentration_by_K_{analysis_type}_{condition}"
    )

    plt.show()
    plt.close()

for analysis_type in AA_ANALYSIS_TYPES:
    for condition in PRIMARY_CONDITIONS:
        plot_concentration_by_K(
            analysis_type,
            condition,
        )


## 4. Aggregate across top-\(m\)

A parent \(K\) is more compelling if concentration is not driven by one arbitrary \(m\).

For each analysis type × condition × parent \(K\), calculate:

- mean \(\log_2 C\);
- median \(\log_2 C\);
- fraction of tested \(m\) values with \(C>1\);
- maximum \(C\);
- number of available \(m\) values.

The mean \(\log_2 C\) is the primary aggregate score.


In [ ]:
# ============================================================
# AGGREGATE ACROSS m
# ============================================================

aggregate_df = (
    concentration_df
    .groupby(
        [
            "analysis_type",
            "condition",
            "K",
            "parent_ratio",
        ],
        as_index=False,
    )
    .agg(
        mean_log2_concentration=(
            "log2_concentration",
            "mean",
        ),
        median_log2_concentration=(
            "log2_concentration",
            "median",
        ),
        mean_concentration_ratio=(
            "concentration_ratio",
            "mean",
        ),
        max_concentration_ratio=(
            "concentration_ratio",
            "max",
        ),
        fraction_C_gt_1=(
            "concentration_ratio",
            lambda x: np.mean(
                np.asarray(x) > 1
            ),
        ),
        n_m=(
            "top_m",
            "count",
        ),
        mean_AA_accuracy=(
            "AA_accuracy",
            "mean",
        ),
    )
)

aggregate_df = aggregate_df[
    aggregate_df["n_m"]
    >= MIN_M_PER_K
].copy()

aggregate_df.to_csv(
    OUTPUT_DIR / "AA_information_concentration_by_K.csv",
    index=False,
)

display(
    aggregate_df
    .sort_values(
        "mean_log2_concentration",
        ascending=False,
    )
    .head(30)
)


In [ ]:
# ============================================================
# AGGREGATE PLOTS
# ============================================================

def plot_aggregate_concentration(
    analysis_type="spatial",
    condition="intact",
):
    s = aggregate_df[
        (aggregate_df["analysis_type"] == analysis_type) &
        (aggregate_df["condition"] == condition)
    ].sort_values("K")

    if len(s) == 0:
        return

    fig, ax = plt.subplots(
        figsize=(8.5, 5.0)
    )

    ax.axhline(
        0,
        color="gray",
        linestyle="--",
        linewidth=1.2,
    )

    ax.plot(
        s["K"],
        s["mean_log2_concentration"],
        marker="o",
        linewidth=2,
        markersize=4,
    )

    # label top 5 parent K values
    top = s.sort_values(
        "mean_log2_concentration",
        ascending=False,
    ).head(5)

    for _, r in top.iterrows():

        ax.annotate(
            f"K={int(r['K'])}",
            (
                r["K"],
                r["mean_log2_concentration"],
            ),
            xytext=(0, 6),
            textcoords="offset points",
            ha="center",
            fontsize=8,
        )

    ax.set_xlabel(
        "Parent MS-AA model order K"
    )

    ax.set_ylabel(
        r"Mean information concentration, $\log_2(q_{\rm eq}/m)$"
    )

    ax.set_title(
        f"{analysis_type} MS-AA | {condition}\n"
        "aggregated across top-m values"
    )

    plt.tight_layout()

    save_current_fig(
        f"aggregate_concentration_{analysis_type}_{condition}"
    )

    plt.show()
    plt.close()

for analysis_type in AA_ANALYSIS_TYPES:
    for condition in PRIMARY_CONDITIONS:
        plot_aggregate_concentration(
            analysis_type,
            condition,
        )


## 5. Joint intact + word score

The most convincing parent scale should ideally generalize across the two stimulus conditions rather than appearing in only one.

For each analysis type and \(K\), we average \(\log_2 C\) across:

- top-\(m\) values, and
- intact + word.

This produces one joint concentration score per parent \(K\).


In [ ]:
# ============================================================
# JOINT SCORE ACROSS INTACT + WORD
# ============================================================

joint_stimulus_df = (
    concentration_df[
        concentration_df["condition"].isin(
            PRIMARY_CONDITIONS
        )
    ]
    .groupby(
        [
            "analysis_type",
            "K",
            "parent_ratio",
        ],
        as_index=False,
    )
    .agg(
        mean_log2_concentration=(
            "log2_concentration",
            "mean",
        ),
        median_log2_concentration=(
            "log2_concentration",
            "median",
        ),
        mean_concentration_ratio=(
            "concentration_ratio",
            "mean",
        ),
        fraction_C_gt_1=(
            "concentration_ratio",
            lambda x: np.mean(
                np.asarray(x) > 1
            ),
        ),
        n_cells=(
            "log2_concentration",
            "count",
        ),
    )
)

joint_stimulus_df.to_csv(
    OUTPUT_DIR / "AA_information_concentration_joint_intact_word.csv",
    index=False,
)

for analysis_type in AA_ANALYSIS_TYPES:

    print("\n", analysis_type.upper())

    display(
        joint_stimulus_df[
            joint_stimulus_df["analysis_type"]
            == analysis_type
        ]
        .sort_values(
            "mean_log2_concentration",
            ascending=False,
        )
        .head(15)
    )


In [ ]:
# ============================================================
# JOINT PLOT
# ============================================================

fig, ax = plt.subplots(
    figsize=(8.7, 5.2)
)

ax.axhline(
    0,
    color="gray",
    linestyle="--",
    linewidth=1.2,
)

for analysis_type, linestyle, marker in [
    ("spatial", "-", "o"),
    ("temporal", "--", "s"),
]:

    s = joint_stimulus_df[
        joint_stimulus_df["analysis_type"]
        == analysis_type
    ].sort_values("parent_ratio")

    ax.plot(
        s["parent_ratio"],
        s["mean_log2_concentration"],
        linestyle=linestyle,
        marker=marker,
        linewidth=2,
        markersize=4,
        label=analysis_type,
    )

    top = s.sort_values(
        "mean_log2_concentration",
        ascending=False,
    ).head(3)

    for _, r in top.iterrows():

        ax.annotate(
            f"K={int(r['K'])}",
            (
                r["parent_ratio"],
                r["mean_log2_concentration"],
            ),
            xytext=(0, 6),
            textcoords="offset points",
            ha="center",
            fontsize=8,
        )

ax.set_xlabel(
    "Normalized parent component ratio"
)

ax.set_ylabel(
    r"Mean $\log_2(q_{\rm eq}/m)$"
)

ax.set_title(
    "Information concentration across intact + word"
)

ax.legend(
    frameon=False
)

plt.tight_layout()

save_current_fig(
    "joint_intact_word_concentration_spatial_temporal"
)

plt.show()
plt.close()


## 6. Permutation test: is there a special parent \(K\)?

This is the main formal test of the hypothesis.

### Null hypothesis

Within each analysis type, condition, and top-\(m\), the concentration values are **not systematically associated with parent model order \(K\)**.

We therefore permute concentration values across \(K\) independently within each condition × top-\(m\) stratum, preserving:

- the observed concentration distribution;
- differences among conditions;
- differences among \(m\);
- the actual set of tested parent \(K\) values.

For each permutation, we recompute the mean concentration score for each parent \(K\) and record the **maximum** score across \(K\).

The empirical maximum is compared against this null distribution.

This controls the "we looked across many K values and picked the biggest one" problem for the global peak.


In [ ]:
# ============================================================
# PERMUTATION TEST FOR GLOBAL K PEAK
# ============================================================

rng = np.random.default_rng(
    RNG_SEED
)

def aggregate_score_from_long(
    df,
    analysis_type,
    conditions=PRIMARY_CONDITIONS,
):

    s = df[
        (df["analysis_type"] == analysis_type) &
        (df["condition"].isin(conditions))
    ].copy()

    out = (
        s.groupby(
            ["K"],
            as_index=False,
        )
        .agg(
            score=(
                "log2_concentration",
                "mean",
            ),
            n=(
                "log2_concentration",
                "count",
            ),
        )
    )

    return out

def permute_within_strata(
    df,
    analysis_type,
    conditions=PRIMARY_CONDITIONS,
    rng=None,
):
    rng = (
        np.random.default_rng()
        if rng is None
        else rng
    )

    s = df[
        (df["analysis_type"] == analysis_type) &
        (df["condition"].isin(conditions))
    ].copy()

    # Shuffle concentration across K within condition x top_m.
    shuffled_parts = []

    for _, g in s.groupby(
        ["condition", "top_m"],
        sort=False,
    ):

        gg = g.copy()

        gg["log2_concentration"] = rng.permutation(
            gg["log2_concentration"].to_numpy()
        )

        shuffled_parts.append(
            gg
        )

    return pd.concat(
        shuffled_parts,
        ignore_index=True,
    )

global_peak_rows = []
global_nulls = {}

for analysis_type in AA_ANALYSIS_TYPES:

    observed = aggregate_score_from_long(
        concentration_df,
        analysis_type,
    )

    if len(observed) == 0:
        continue

    best_row = observed.loc[
        observed["score"].idxmax()
    ]

    observed_max = float(
        best_row["score"]
    )

    observed_best_K = int(
        best_row["K"]
    )

    null_max = np.empty(
        N_PERM,
        dtype=float,
    )

    for i in range(N_PERM):

        perm = permute_within_strata(
            concentration_df,
            analysis_type,
            rng=rng,
        )

        agg = aggregate_score_from_long(
            perm,
            analysis_type,
        )

        null_max[i] = (
            agg["score"].max()
        )

    p_global = (
        1
        + np.sum(
            null_max
            >= observed_max
        )
    ) / (
        N_PERM + 1
    )

    global_nulls[
        analysis_type
    ] = null_max

    global_peak_rows.append({
        "analysis_type": analysis_type,
        "observed_best_K": observed_best_K,
        "observed_max_mean_log2_concentration": observed_max,
        "null_mean_max": float(
            np.mean(null_max)
        ),
        "null_95pct_max": float(
            np.quantile(
                null_max,
                0.95,
            )
        ),
        "global_peak_p": p_global,
    })

global_peak_test_df = pd.DataFrame(
    global_peak_rows
)

display(
    global_peak_test_df
)

global_peak_test_df.to_csv(
    OUTPUT_DIR / "global_parentK_peak_permutation_test.csv",
    index=False,
)


In [ ]:
# ============================================================
# GLOBAL NULL PLOTS
# ============================================================

for analysis_type in global_nulls:

    null = global_nulls[
        analysis_type
    ]

    obs = global_peak_test_df[
        global_peak_test_df["analysis_type"]
        == analysis_type
    ].iloc[0]

    fig, ax = plt.subplots(
        figsize=(7.0, 4.6)
    )

    ax.hist(
        null,
        bins=40,
        alpha=0.75,
    )

    ax.axvline(
        obs[
            "observed_max_mean_log2_concentration"
        ],
        linewidth=2,
        linestyle="--",
        label=(
            f"observed max, K="
            f"{int(obs['observed_best_K'])}"
        ),
    )

    ax.set_xlabel(
        "Maximum mean log2 concentration under null"
    )

    ax.set_ylabel(
        "Permutation count"
    )

    ax.set_title(
        f"{analysis_type}: global parent-K peak test\n"
        f"p={obs['global_peak_p']:.4f}"
    )

    ax.legend(
        frameon=False
    )

    plt.tight_layout()

    save_current_fig(
        f"global_peak_null_{analysis_type}"
    )

    plt.show()
    plt.close()


## 7. Pre-specified candidate-\(K\) permutation tests

The global peak test asks whether **some** parent \(K\) is unusually concentrated.

Because spatial \(K\approx50\) was identified from the earlier top-\(m\) decoding analyses rather than from the current concentration statistic, it can also be evaluated as a **pre-specified candidate**.

For a candidate \(K_0\), the observed score is the mean \(\log_2 C\) across intact, word, and available top-\(m\) values at \(K_0\).

The permutation null shuffles concentration across parent \(K\) within condition × \(m\), then asks how often the shuffled value at the same candidate \(K_0\) is at least as large as observed.


In [ ]:
# ============================================================
# SPECIFIC-K PERMUTATION TEST
# ============================================================

def candidate_K_score(
    df,
    analysis_type,
    K0,
    conditions=PRIMARY_CONDITIONS,
):

    s = df[
        (df["analysis_type"] == analysis_type) &
        (df["condition"].isin(conditions)) &
        (df["K"] == K0)
    ]

    if len(s) == 0:
        return np.nan

    return float(
        s["log2_concentration"].mean()
    )

candidate_rows = []

for analysis_type, candidate_Ks in CANDIDATE_K_BY_ANALYSIS.items():

    available_Ks = set(
        concentration_df[
            concentration_df["analysis_type"]
            == analysis_type
        ]["K"].unique()
    )

    for K0 in candidate_Ks:

        if K0 not in available_Ks:
            print(
                "Skipping unavailable candidate:",
                analysis_type,
                K0,
            )
            continue

        observed = candidate_K_score(
            concentration_df,
            analysis_type,
            K0,
        )

        null = np.empty(
            N_PERM,
            dtype=float,
        )

        for i in range(N_PERM):

            perm = permute_within_strata(
                concentration_df,
                analysis_type,
                rng=rng,
            )

            null[i] = candidate_K_score(
                perm,
                analysis_type,
                K0,
            )

        p = (
            1
            + np.sum(
                null >= observed
            )
        ) / (
            N_PERM + 1
        )

        candidate_rows.append({
            "analysis_type": analysis_type,
            "candidate_K": K0,
            "observed_mean_log2_concentration": observed,
            "observed_mean_ratio": float(
                2 ** observed
            ),
            "null_mean": float(
                np.mean(null)
            ),
            "null_95pct": float(
                np.quantile(
                    null,
                    0.95,
                )
            ),
            "permutation_p": p,
        })

candidate_test_df = pd.DataFrame(
    candidate_rows
)

display(
    candidate_test_df
)

candidate_test_df.to_csv(
    OUTPUT_DIR / "candidate_parentK_permutation_tests.csv",
    index=False,
)


## 8. Local enhancement test

A global score can be high simply because concentration changes gradually with \(K\).

To test for a **localized bump**, compute:

\[
L(K)=
S(K)
-
\frac{S(K_{\mathrm{lower}})+S(K_{\mathrm{upper}})}{2},
\]

where \(S(K)\) is mean \(\log_2 C\), and the lower/upper values are the nearest tested parent model orders.

Positive \(L(K)\) means the candidate is elevated relative to its immediate local scale neighborhood.

This is particularly useful for spatial \(K=50\).


In [ ]:
# ============================================================
# LOCAL ENHANCEMENT
# ============================================================

def local_enhancement_table(
    analysis_type,
    conditions=PRIMARY_CONDITIONS,
):

    agg = aggregate_score_from_long(
        concentration_df,
        analysis_type,
        conditions=conditions,
    ).sort_values("K").reset_index(drop=True)

    rows = []

    for i in range(
        1,
        len(agg) - 1,
    ):

        cur = agg.iloc[i]
        lo = agg.iloc[i - 1]
        hi = agg.iloc[i + 1]

        neighbor_mean = 0.5 * (
            lo["score"]
            + hi["score"]
        )

        rows.append({
            "analysis_type": analysis_type,
            "K": int(cur["K"]),
            "score": float(cur["score"]),
            "lower_K": int(lo["K"]),
            "upper_K": int(hi["K"]),
            "neighbor_mean": float(neighbor_mean),
            "local_enhancement": float(
                cur["score"]
                - neighbor_mean
            ),
        })

    return pd.DataFrame(
        rows
    )

local_parts = [
    local_enhancement_table(a)
    for a in AA_ANALYSIS_TYPES
]

local_df = pd.concat(
    local_parts,
    ignore_index=True,
)

local_df.to_csv(
    OUTPUT_DIR / "local_parentK_enhancement.csv",
    index=False,
)

for analysis_type in AA_ANALYSIS_TYPES:

    print("\n", analysis_type.upper())

    display(
        local_df[
            local_df["analysis_type"]
            == analysis_type
        ]
        .sort_values(
            "local_enhancement",
            ascending=False,
        )
        .head(15)
    )


In [ ]:
# ============================================================
# PERMUTATION TEST FOR LOCAL ENHANCEMENT AT CANDIDATES
# ============================================================

def local_enhancement_at_K(
    df,
    analysis_type,
    K0,
    conditions=PRIMARY_CONDITIONS,
):

    agg = aggregate_score_from_long(
        df,
        analysis_type,
        conditions=conditions,
    ).sort_values("K").reset_index(drop=True)

    where = np.where(
        agg["K"].to_numpy()
        == K0
    )[0]

    if len(where) == 0:
        return np.nan

    i = int(
        where[0]
    )

    if (
        i == 0
        or i == len(agg) - 1
    ):
        return np.nan

    cur = float(
        agg.iloc[i]["score"]
    )

    neighbor = 0.5 * (
        float(
            agg.iloc[i - 1]["score"]
        )
        + float(
            agg.iloc[i + 1]["score"]
        )
    )

    return cur - neighbor

local_test_rows = []

for analysis_type, candidate_Ks in CANDIDATE_K_BY_ANALYSIS.items():

    for K0 in candidate_Ks:

        observed = local_enhancement_at_K(
            concentration_df,
            analysis_type,
            K0,
        )

        if not np.isfinite(observed):
            continue

        null = np.empty(
            N_PERM,
            dtype=float,
        )

        for i in range(N_PERM):

            perm = permute_within_strata(
                concentration_df,
                analysis_type,
                rng=rng,
            )

            null[i] = local_enhancement_at_K(
                perm,
                analysis_type,
                K0,
            )

        null = null[
            np.isfinite(null)
        ]

        p = (
            1
            + np.sum(
                null >= observed
            )
        ) / (
            len(null) + 1
        )

        local_test_rows.append({
            "analysis_type": analysis_type,
            "candidate_K": K0,
            "observed_local_enhancement": observed,
            "null_mean": float(
                np.mean(null)
            ),
            "null_95pct": float(
                np.quantile(
                    null,
                    0.95,
                )
            ),
            "local_permutation_p": p,
        })

local_candidate_test_df = pd.DataFrame(
    local_test_rows
)

display(
    local_candidate_test_df
)

local_candidate_test_df.to_csv(
    OUTPUT_DIR / "candidate_local_enhancement_tests.csv",
    index=False,
)


## 9. Is the concentration effect simply monotonic with parent \(K\)?

If concentration just rises with parent model order, then a "special intermediate scale" interpretation is weak.

We therefore calculate Spearman correlation between parent \(K\) and concentration score.

A strong positive monotonic correlation suggests progressive concentration with increasing \(K\).

A weak correlation combined with a significant local/global peak supports a scale-specific regime.


In [ ]:
# ============================================================
# MONOTONIC TREND TEST
# ============================================================

trend_rows = []

for analysis_type in AA_ANALYSIS_TYPES:

    for condition in PRIMARY_CONDITIONS:

        s = aggregate_df[
            (aggregate_df["analysis_type"] == analysis_type) &
            (aggregate_df["condition"] == condition)
        ].sort_values("K")

        if len(s) < 3:
            continue

        rho, p = spearmanr(
            s["K"],
            s["mean_log2_concentration"],
            nan_policy="omit",
        )

        trend_rows.append({
            "analysis_type": analysis_type,
            "condition": condition,
            "spearman_rho_K_vs_concentration": rho,
            "p_value": p,
            "n_K": len(s),
        })

trend_df = pd.DataFrame(
    trend_rows
)

display(
    trend_df
)

trend_df.to_csv(
    OUTPUT_DIR / "monotonic_K_concentration_trends.csv",
    index=False,
)


## 10. Stability across top-\(m\)

If one \(K\) is genuinely a concentration regime, it should score highly for multiple values of \(m\), not just one.

The heatmap below makes that immediately visible.


In [ ]:
# ============================================================
# HEATMAP K x top-m
# ============================================================

def plot_concentration_heatmap(
    analysis_type="spatial",
    condition="intact",
):

    s = concentration_df[
        (concentration_df["analysis_type"] == analysis_type) &
        (concentration_df["condition"] == condition)
    ].copy()

    piv = s.pivot_table(
        index="top_m",
        columns="K",
        values="log2_concentration",
        aggfunc="mean",
    )

    piv = piv.sort_index()
    piv = piv.reindex(
        sorted(piv.columns),
        axis=1,
    )

    if piv.size == 0:
        return

    vmax = np.nanmax(
        np.abs(piv.to_numpy())
    )

    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1

    fig, ax = plt.subplots(
        figsize=(10.0, 4.5)
    )

    im = ax.imshow(
        piv.to_numpy(),
        aspect="auto",
        cmap="coolwarm",
        vmin=-vmax,
        vmax=vmax,
    )

    ax.set_xticks(
        np.arange(
            len(piv.columns)
        )
    )

    ax.set_xticklabels(
        [str(k) for k in piv.columns],
        rotation=45,
        ha="right",
    )

    ax.set_yticks(
        np.arange(
            len(piv.index)
        )
    )

    ax.set_yticklabels(
        [str(m) for m in piv.index]
    )

    ax.set_xlabel(
        "Parent MS-AA K"
    )

    ax.set_ylabel(
        "top-m"
    )

    ax.set_title(
        f"{analysis_type} | {condition}\n"
        r"$\log_2(q_{\rm eq}/m)$"
    )

    cbar = plt.colorbar(
        im,
        ax=ax,
    )

    cbar.set_label(
        r"$\log_2$ concentration"
    )

    plt.tight_layout()

    save_current_fig(
        f"concentration_heatmap_{analysis_type}_{condition}"
    )

    plt.show()
    plt.close()

for condition in PRIMARY_CONDITIONS:
    plot_concentration_heatmap(
        "spatial",
        condition,
    )


## 11. Candidate \(K=50\) detailed profile

Because spatial \(K=50\) was motivated by the prior top-\(m\) decoding analyses, this table shows exactly how its concentration behaves across conditions and \(m\).


In [ ]:
# ============================================================
# SPATIAL K=50 DETAIL
# ============================================================

spatial_K50_df = concentration_df[
    (concentration_df["analysis_type"] == "spatial") &
    (concentration_df["K"] == PRIMARY_SPATIAL_K)
].copy()

display(
    spatial_K50_df[[
        "condition",
        "K",
        "top_m",
        "AA_accuracy",
        "pca_equivalent_dimension",
        "concentration_ratio",
        "log2_concentration",
    ]].sort_values([
        "condition",
        "top_m",
    ])
)

spatial_K50_df.to_csv(
    OUTPUT_DIR / f"spatial_K{PRIMARY_SPATIAL_K}_detailed_concentration.csv",
    index=False,
)


## 12. Manuscript-ready summary

This cell prints conservative interpretations based on the actual statistical output.

It does **not** automatically claim a special intermediate regime. The wording changes depending on whether the global peak, candidate-\(K\), and local-enhancement tests are supported.


In [ ]:
# ============================================================
# AUTOMATIC CONSERVATIVE SUMMARY
# ============================================================

def format_p(p):
    if pd.isna(p):
        return "NA"
    if p < 0.001:
        return "<0.001"
    return f"{p:.3f}"

print("=" * 72)
print("GLOBAL PEAK TESTS")
print("=" * 72)

for _, r in global_peak_test_df.iterrows():

    supported = (
        r["global_peak_p"] < 0.05
    )

    print(
        f"{r['analysis_type']}: "
        f"best K={int(r['observed_best_K'])}, "
        f"mean log2 concentration="
        f"{r['observed_max_mean_log2_concentration']:.3f}, "
        f"global permutation p="
        f"{format_p(r['global_peak_p'])}."
    )

    if supported:
        print(
            "  -> Evidence that concentration is non-uniform across parent K."
        )
    else:
        print(
            "  -> No corrected evidence that the maximum K-level concentration "
            "exceeds what could arise by searching across K."
        )

print("\n" + "=" * 72)
print("PRE-SPECIFIED CANDIDATES")
print("=" * 72)

for _, r in candidate_test_df.iterrows():

    print(
        f"{r['analysis_type']} K={int(r['candidate_K'])}: "
        f"mean ratio≈{r['observed_mean_ratio']:.2f}, "
        f"permutation p={format_p(r['permutation_p'])}."
    )

print("\n" + "=" * 72)
print("LOCAL ENHANCEMENT")
print("=" * 72)

for _, r in local_candidate_test_df.iterrows():

    print(
        f"{r['analysis_type']} K={int(r['candidate_K'])}: "
        f"local enhancement={r['observed_local_enhancement']:.3f}, "
        f"p={format_p(r['local_permutation_p'])}."
    )


# Decision framework

## Strong support for the hypothesis

The strongest pattern would be:

1. the global parent-\(K\) peak permutation test is significant;
2. the peak falls near a model order already implicated by the earlier top-\(m\) analyses;
3. the pre-specified spatial \(K=50\) test is significant or strongly elevated;
4. local enhancement at \(K=50\) is positive and significant;
5. the effect occurs across several top-\(m\) values;
6. intact and word both contribute;
7. the concentration curve is not simply monotonically increasing with \(K\).

Then the paper could reasonably say:

> Sparse information was not distributed uniformly across archetypal model orders. Instead, particular parent decompositions concentrated cross-subject temporal information into small archetype subsets: \(m\) selected archetypes from these models preserved information comparable to substantially larger conventional PCA representations. This concentration was strongest at [K], suggesting that the geometry of the parent archetypal representation determines how efficiently task-relevant information is captured by a small number of motifs.

## Partial support

If \(C>1\) occurs broadly but the global/local peak tests are not significant:

> Selected archetype subsets frequently preserved information comparable to PCA representations with more dimensions than the number of retained archetypes, but this information concentration was not uniquely localized to a single parent model order.

That is still useful and avoids overclaiming an intermediate regime.

## No support

If \(C\leq1\) for most configurations:

> PCA provided a more compact representation in the conventional information-preservation sense. The contribution of MS-AA therefore lies primarily in its data-anchored geometry and network interpretability rather than superior compression efficiency.

That outcome would still answer the PCA reviewer criticism cleanly.

# Confirmatory extension

Because the current AA top-\(m\) subsets were selected with the same decoding statistic used for evaluation, the final confirmatory analysis should repeat this notebook after **held-out archetype selection**. The concentration statistic and permutation framework can then remain unchanged.
